# LeetCode 229 — Majority Element II bằng Python

Link đề: https://leetcode.com/problems/majority-element-ii/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Counting, Sorting, Boyer-Moore Voting

Skeleton LeetCode:

```python
class Solution:
    def majorityElement(self, nums: List[int]) -> List[int]:
        ...
```

Ghi chú: đề, ví dụ và constraints lấy từ đề gốc; mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với một lời giải brute force độc lập (đếm số lần xuất hiện của từng giá trị bằng `list.count`).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm **mọi** giá trị xuất hiện **nhiều hơn `⌊n/3⌋` lần** trong mảng `nums` độ dài `n` (thứ tự kết quả tùy ý).
2. Chứng minh được một **quan sát then chốt**: có **tối đa 2** giá trị như vậy (nếu có 3 giá trị thì tổng số lần xuất hiện vượt `n`).
3. Đi từ **đếm từng giá trị bằng `count` (`O(n·d)`)** → **`Counter` / hash map (`O(n)` thời gian, `O(n)` bộ nhớ)** → **sắp xếp và đếm đoạn bằng nhau** → **mẹo `nums[i] == nums[i + n//3]` trên mảng đã sắp** → **Boyer–Moore với hai ứng viên (`O(n)` thời gian, `O(1)` bộ nhớ)**.
4. Hiểu thuật toán **bỏ phiếu Boyer–Moore mở rộng**: giữ hai ứng viên và hai bộ đếm, "triệt tiêu" ba giá trị khác nhau cùng lúc, rồi **bắt buộc kiểm tra lại** bằng lượt đếm thứ hai.
5. Tránh các lỗi kinh điển: nhầm với bài "Majority Element" (`> n/2`), quên lượt xác minh, dùng `>=` thay vì `>`, và kiểm tra `m == m2` **sau** khi đã xét `n1 == 0` (sai thứ tự nhánh).

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` kích thước `n`. Tìm **tất cả** các phần tử xuất hiện **nhiều hơn `⌊n / 3⌋` lần** (`⌊x⌋` là phần nguyên dưới của `x`). Thứ tự kết quả không quan trọng.

**Ví dụ 1**

- Input: `nums = [3,2,3]`
- Output: `[3]`
- Giải thích: `n = 3`, `⌊3/3⌋ = 1`; `3` xuất hiện 2 lần `> 1`, còn `2` xuất hiện 1 lần (không `> 1`).

**Ví dụ 2**

- Input: `nums = [1]`
- Output: `[1]`
- Giải thích: `n = 1`, `⌊1/3⌋ = 0`; `1` xuất hiện 1 lần `> 0`.

**Ví dụ 3**

- Input: `nums = [1,2]`
- Output: `[1,2]`
- Giải thích: `n = 2`, `⌊2/3⌋ = 0`; cả `1` và `2` đều xuất hiện 1 lần `> 0`.

**Ràng buộc:** `1 <= nums.length <= 5 * 10⁴`; `-10⁹ <= nums[i] <= 10⁹`.

**Follow up:** giải trong thời gian tuyến tính và bộ nhớ `O(1)`.

In [1]:
from typing import List
import random
from collections import Counter

examples = [
    ([3, 2, 3], [3]),
    ([1], [1]),
    ([1, 2], [1, 2]),
]
for nums, expected in examples:
    print(f"nums={nums}, n // 3 = {len(nums) // 3}, kỳ vọng {expected}")

nums=[3, 2, 3], n // 3 = 1, kỳ vọng [3]
nums=[1], n // 3 = 0, kỳ vọng [1]
nums=[1, 2], n // 3 = 0, kỳ vọng [1, 2]


## 3. Kiến thức nền

### 3.1. Chia nguyên `//` và "nhiều hơn `⌊n/3⌋` lần"

`n // 3` là phép chia **làm tròn xuống**. Điều kiện của đề là `số_lần > n // 3`. Vì số lần là số nguyên nên `số_lần > n // 3` cũng tương đương `số_lần > n / 3` (phép chia thực): ví dụ `n = 5` thì `n // 3 = 1` và `n / 3 ≈ 1.67`, số lần `2` thỏa cả hai, số lần `1` không thỏa cả hai.

In [2]:
for n in [1, 2, 3, 4, 5, 6, 7, 9]:
    print(f"n={n}: n // 3 = {n // 3}, n / 3 = {n / 3:.2f}, cần số lần >= {n // 3 + 1}")
    for count in range(0, n + 1):
        assert (count > n // 3) == (count > n / 3)

n=1: n // 3 = 0, n / 3 = 0.33, cần số lần >= 1
n=2: n // 3 = 0, n / 3 = 0.67, cần số lần >= 1
n=3: n // 3 = 1, n / 3 = 1.00, cần số lần >= 2
n=4: n // 3 = 1, n / 3 = 1.33, cần số lần >= 2
n=5: n // 3 = 1, n / 3 = 1.67, cần số lần >= 2
n=6: n // 3 = 2, n / 3 = 2.00, cần số lần >= 3
n=7: n // 3 = 2, n / 3 = 2.33, cần số lần >= 3
n=9: n // 3 = 3, n / 3 = 3.00, cần số lần >= 4


### 3.2. `list.count` và `Counter`

`nums.count(x)` quét cả mảng và trả số lần `x` xuất hiện (`O(n)` mỗi lần gọi). `Counter(nums)` đếm **mọi** giá trị trong **một** lượt, cho từ điển `giá trị -> số lần`; `.items()` duyệt các cặp.

In [3]:
nums = [1, 1, 1, 3, 3, 2, 2, 2]
print(nums.count(1), nums.count(3), nums.count(9))        # 3 2 0
cnt = Counter(nums)
print(cnt)                                                 # Counter({1: 3, 2: 3, 3: 2})
print({x: c for x, c in cnt.items() if c > len(nums) // 3})

3 2 0
Counter({1: 3, 2: 3, 3: 2})
{1: 3, 2: 3}


### 3.3. Tập hợp `set` để bỏ trùng

Kết quả không được chứa phần tử trùng lặp (một giá trị chỉ cần xuất hiện một lần trong đáp án). `set(...)` bỏ trùng, `sorted(set(...))` cho list đã sắp (tiện so sánh khi test vì thứ tự đáp án không quan trọng).

In [4]:
cands = [2, 2]
print(cands, set(cands), sorted(set(cands)))
print(sorted([3, 1, 2]) == sorted([2, 3, 1]))               # so sánh bỏ qua thứ tự

[2, 2] {2} [2]
True


### 3.4. Sắp xếp và "đoạn bằng nhau"

Sau khi `sorted(nums)`, các giá trị bằng nhau nằm **liền nhau** thành các đoạn. Số lần xuất hiện của một giá trị bằng độ dài đoạn của nó. Ta duyệt và đếm độ dài đoạn liên tiếp.

In [5]:
s = sorted([3, 1, 2, 1, 3, 1])
print(s)                                   # [1, 1, 1, 2, 3, 3]
runs = []
i = 0
while i < len(s):
    j = i
    while j < len(s) and s[j] == s[i]:
        j += 1
    runs.append((s[i], j - i))             # (giá trị, độ dài đoạn)
    i = j
print(runs)                                # [(1, 3), (2, 1), (3, 2)]

[1, 1, 1, 2, 3, 3]
[(1, 3), (2, 1), (3, 2)]


### 3.5. Ý tưởng "bỏ phiếu" Boyer–Moore (ôn lại bài `> n/2`)

Với bài "phần tử xuất hiện **hơn nửa**" (LeetCode 169): giữ một ứng viên và một bộ đếm; gặp cùng ứng viên thì `+1`, khác thì `−1`; bộ đếm về `0` thì đổi ứng viên. Hiểu: mỗi cặp hai phần tử **khác nhau** "triệt tiêu" nhau; phần tử chiếm hơn nửa không thể bị triệt tiêu hết.

In [6]:
def majority_half(nums):
    cand, cnt = None, 0
    for x in nums:
        if cnt == 0:
            cand, cnt = x, 1
        elif x == cand:
            cnt += 1
        else:
            cnt -= 1
    return cand


print(majority_half([2, 2, 1, 1, 1, 2, 2]))      # 2
print(majority_half([3, 3, 4]))                  # 3

2
3


Với ngưỡng `n/3`, ta cần **hai** ứng viên, và mỗi lần "triệt tiêu" gồm **ba giá trị đôi một khác nhau** (hai ứng viên và phần tử mới). Ta sẽ làm kỹ ở các cách sau.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

**Có tối đa bao nhiêu đáp án?** Nếu có ba giá trị khác nhau cùng xuất hiện `> n/3` lần thì tổng số phần tử `> 3·(n/3) = n`: vô lý. Vậy đáp án có **tối đa 2** phần tử.

Tính tay ví dụ `nums = [1,1,1,3,3,2,2,2]`, `n = 8`, `n // 3 = 2`:

| giá trị | số lần | `> 2`? |
|---|---|---|
| `1` | 3 | có |
| `3` | 2 | không |
| `2` | 3 | có |

Đáp án `[1, 2]`. Các giá trị chỉ cần đếm số lần; bất kỳ cách nào đếm được số lần mỗi giá trị đều giải được bài. Câu hỏi là: có thể làm với `O(1)` bộ nhớ không (follow up)?

In [7]:
nums = [1, 1, 1, 3, 3, 2, 2, 2]
n = len(nums)
print("n =", n, "| ngưỡng: số lần >", n // 3)
for x in sorted(set(nums)):
    c = nums.count(x)
    print(f"  giá trị {x}: {c} lần ->", "THỎA" if c > n // 3 else "không")

n = 8 | ngưỡng: số lần > 2
  giá trị 1: 3 lần -> THỎA
  giá trị 2: 3 lần -> THỎA
  giá trị 3: 2 lần -> không


## 5. Cách 1 — Brute Force / baseline: đếm từng giá trị bằng `count`

**Ý tưởng.** Với mỗi giá trị **khác nhau** trong `nums`, gọi `nums.count(x)` rồi so với `n // 3`. Duyệt `set(nums)` để mỗi giá trị chỉ xét một lần.

In [8]:
def majority_bruteforce(nums: list[int]) -> list[int]:
    n = len(nums)
    result = []
    for x in set(nums):                     # mỗi giá trị khác nhau xét một lần
        if nums.count(x) > n // 3:          # count quét cả mảng: O(n)
            result.append(x)
    return result


for nums, expected in examples:
    got = majority_bruteforce(nums)
    print(f"nums={nums} -> {sorted(got)} (kỳ vọng {sorted(expected)})")

nums=[3, 2, 3] -> [3] (kỳ vọng [3])
nums=[1] -> [1] (kỳ vọng [1])
nums=[1, 2] -> [1, 2] (kỳ vọng [1, 2])


**Độ phức tạp (Cách 1).** Gọi `n = len(nums)` và `d` là số giá trị khác nhau. Mỗi giá trị `count` mất `O(n)`: **Time `O(n·d)`**, tệ nhất `O(n²)` (khi mọi giá trị khác nhau). **Space `O(d)`** cho `set`. Với `n = 5·10⁴` có thể lên tới `2.5·10⁹` thao tác: **quá chậm**.

**Nút thắt.** Ta quét lại cả mảng cho **mỗi** giá trị. Nên đếm tất cả giá trị trong **một** lượt.

## 6. Ý tưởng thuật toán / bước đột phá

**Ý 1 — đếm một lượt.** Dùng từ điển `giá trị → số lần`: một lượt duyệt `O(n)`, bộ nhớ `O(d)`.

**Ý 2 — chỉ có tối đa 2 ứng viên (follow up `O(1)` bộ nhớ).** Vì đáp án có tối đa 2 phần tử, ta chỉ cần **tìm tối đa 2 ứng viên** rồi **xác minh** số lần của chúng bằng một lượt đếm nữa.

Cách tìm ứng viên (Boyer–Moore mở rộng): giữ `(m1, n1)` và `(m2, n2)` là hai ứng viên và "số phiếu" còn lại của chúng. Với mỗi `x`:

1. `x == m1` → `n1 += 1`;
2. ngược lại `x == m2` → `n2 += 1`;
3. ngược lại nếu `n1 == 0` → `m1, n1 = x, 1`;
4. ngược lại nếu `n2 == 0` → `m2, n2 = x, 1`;
5. ngược lại (`x` khác cả hai ứng viên, cả hai còn phiếu) → `n1 −= 1`, `n2 −= 1`.

**Vì sao đúng?** Ở bước 5, ba giá trị `x, m1, m2` **đôi một khác nhau** cùng bị "gạch bỏ" một bản mỗi giá trị. Một giá trị xuất hiện `> n/3` lần không thể bị gạch hết: mỗi lần gạch loại tối đa 1 bản của nó trong **ba** phần tử bị loại; nếu mọi lần gạch đều dùng đến nó thì số lần gạch `≤ n/3`, nhưng nó có `> n/3` bản, nên còn ít nhất một bản sống sót làm ứng viên. Vì có thể xuất hiện ứng viên **không** thật sự vượt ngưỡng, **phải xác minh** ở lượt hai.

In [9]:
nums = [1, 1, 1, 3, 3, 2, 2, 2]
n1 = n2 = 0
m1, m2 = 0, 1
for x in nums:
    if x == m1:
        n1 += 1
    elif x == m2:
        n2 += 1
    elif n1 == 0:
        m1, n1 = x, 1
    elif n2 == 0:
        m2, n2 = x, 1
    else:
        n1, n2 = n1 - 1, n2 - 1
    print(f"x={x}: (m1,n1)=({m1},{n1}), (m2,n2)=({m2},{n2})")

x=1: (m1,n1)=(0,0), (m2,n2)=(1,1)
x=1: (m1,n1)=(0,0), (m2,n2)=(1,2)
x=1: (m1,n1)=(0,0), (m2,n2)=(1,3)
x=3: (m1,n1)=(3,1), (m2,n2)=(1,3)
x=3: (m1,n1)=(3,2), (m2,n2)=(1,3)
x=2: (m1,n1)=(3,1), (m2,n2)=(1,2)
x=2: (m1,n1)=(3,0), (m2,n2)=(1,1)
x=2: (m1,n1)=(2,1), (m2,n2)=(1,1)


## 7. Các cách tiếp theo

### Cách 2 — `Counter` (hash map)

**Giải thích.** `cnt = Counter(nums)`; trả về các khóa có số lần `> n // 3`.

In [10]:
def majority_counter(nums: list[int]) -> list[int]:
    n = len(nums)
    cnt = Counter(nums)                         # một lượt đếm mọi giá trị
    return [x for x, c in cnt.items() if c > n // 3]


for nums, expected in examples:
    print(f"nums={nums} -> {sorted(majority_counter(nums))} (kỳ vọng {sorted(expected)})")

nums=[3, 2, 3] -> [3] (kỳ vọng [3])
nums=[1] -> [1] (kỳ vọng [1])
nums=[1, 2] -> [1, 2] (kỳ vọng [1, 2])


In [11]:
# Code nộp LeetCode cho Cách 2
from collections import Counter
from typing import List


class Solution:
    def majorityElement(self, nums: List[int]) -> List[int]:
        n = len(nums)
        return [x for x, c in Counter(nums).items() if c > n // 3]

**Độ phức tạp Cách 2:** **Time `O(n)`** (hash trung bình `O(1)`), **Space `O(d)`** với `d ≤ n` là số giá trị khác nhau; tối đa `O(n)`.

### Cách 3 — Sắp xếp rồi đếm độ dài đoạn bằng nhau

**Giải thích.** Sắp `nums`; duyệt từng đoạn giá trị bằng nhau, độ dài đoạn là số lần; thêm giá trị vào kết quả nếu độ dài `> n // 3`. Không dùng từ điển.

In [12]:
def majority_sort_runs(nums: list[int]) -> list[int]:
    s = sorted(nums)
    n = len(s)
    result = []
    i = 0
    while i < n:
        j = i
        while j < n and s[j] == s[i]:
            j += 1                              # kết thúc đoạn [i, j)
        if j - i > n // 3:
            result.append(s[i])
        i = j
    return result


for nums, expected in examples:
    print(f"nums={nums} -> {sorted(majority_sort_runs(nums))} (kỳ vọng {sorted(expected)})")

nums=[3, 2, 3] -> [3] (kỳ vọng [3])
nums=[1] -> [1] (kỳ vọng [1])
nums=[1, 2] -> [1, 2] (kỳ vọng [1, 2])


**Độ phức tạp Cách 3:** **Time `O(n log n)`** (sắp xếp), **Space `O(n)`** (bản sao `s`; nếu sắp tại chỗ `nums.sort()` là `O(1)` thêm, trừ ngăn xếp sắp xếp).

### Cách 4 — Mẹo `s[i] == s[i + n//3]` trên mảng đã sắp

**Giải thích.** Trong mảng đã sắp `s`, giá trị `x` xuất hiện **ít nhất `k = n//3 + 1` lần** khi và chỉ khi tồn tại `i` với `s[i] == s[i + k − 1] == x`, tức `s[i] == s[i + n//3]` (hai vị trí cách nhau `n//3`, cùng giá trị thì cả đoạn giữa cũng cùng giá trị). Dùng `set` để bỏ trùng.

In [13]:
def majority_sorted_window(nums: list[int]) -> list[int]:
    s = sorted(nums)
    n = len(s)
    step = n // 3
    result = set()
    for i in range(n - step):                   # i + step <= n - 1
        if s[i] == s[i + step]:                 # đoạn s[i..i+step] gồm step + 1 phần tử cùng giá trị
            result.add(s[i])
    return list(result)


for nums, expected in examples:
    print(f"nums={nums} -> {sorted(majority_sorted_window(nums))} (kỳ vọng {sorted(expected)})")

nums=[3, 2, 3] -> [3] (kỳ vọng [3])
nums=[1] -> [1] (kỳ vọng [1])
nums=[1, 2] -> [1, 2] (kỳ vọng [1, 2])


**Độ phức tạp Cách 4:** **Time `O(n log n)`**, **Space `O(n)`**. Ưu điểm: ngắn gọn, không dùng từ điển. Lưu ý khi `step = 0` (`n < 3`): `s[i] == s[i]` luôn đúng, nên mọi phần tử đều vào kết quả, đúng với đề (ví dụ 3).

### Cách 5 — Boyer–Moore hai ứng viên (`O(n)` thời gian, `O(1)` bộ nhớ, đáp án follow up)

**Giải thích.** Lượt 1: tìm hai ứng viên bằng 5 nhánh ở mục 6. Lượt 2: xác minh bằng `nums.count` (hoặc đếm lại bằng hai biến) và giữ những ứng viên có số lần `> n // 3`. Hai ứng viên khởi đầu khác nhau (`0` và `1`) với bộ đếm `0` — vì bộ đếm `0` nên giá trị khởi đầu không có tác dụng gì, chỉ cần `m1 != m2` để nhánh `x == m2` không trùng nhánh `x == m1`.

In [14]:
def majority_boyer_moore(nums: list[int]) -> list[int]:
    n1 = n2 = 0
    m1, m2 = 0, 1                                # hai ứng viên khác nhau, bộ đếm 0
    for x in nums:
        if x == m1:
            n1 += 1
        elif x == m2:
            n2 += 1
        elif n1 == 0:
            m1, n1 = x, 1
        elif n2 == 0:
            m2, n2 = x, 1
        else:
            n1, n2 = n1 - 1, n2 - 1              # gạch ba giá trị khác nhau
    # lượt 2: xác minh thực sự
    c1 = c2 = 0
    for x in nums:
        if x == m1:
            c1 += 1
        elif x == m2:
            c2 += 1
    limit = len(nums) // 3
    result = []
    if c1 > limit:
        result.append(m1)
    if c2 > limit:
        result.append(m2)
    return result


for nums, expected in examples:
    print(f"nums={nums} -> {sorted(majority_boyer_moore(nums))} (kỳ vọng {sorted(expected)})")

nums=[3, 2, 3] -> [3] (kỳ vọng [3])
nums=[1] -> [1] (kỳ vọng [1])
nums=[1, 2] -> [1, 2] (kỳ vọng [1, 2])


**Độ phức tạp Cách 5:** **Time `O(n)`** (hai lượt duyệt), **Space `O(1)`** (bốn biến cho ứng viên/bộ đếm, hai biến đếm xác minh, không dùng cấu trúc dữ liệu phụ).

> Cách 2 là cách nộp ngắn nhất. Cách 5 đáp ứng follow up. Cách 1 và Cách 3 là chuẩn đối chiếu đơn giản.

## 8. TRACE từng bước

Ta theo dõi Boyer–Moore trên `nums = [3, 1, 1, 2, 1, 2, 3, 2]` (đáp án `[1, 2]`) — ví dụ đủ để thấy các nhánh thay đổi ứng viên và gạch phiếu.

In [15]:
def trace_boyer_moore(nums):
    print("nums =", nums, "| n // 3 =", len(nums) // 3)
    n1 = n2 = 0
    m1, m2 = 0, 1
    for i, x in enumerate(nums):
        if x == m1:
            n1 += 1
            action = f"trùng m1 -> n1 = {n1}"
        elif x == m2:
            n2 += 1
            action = f"trùng m2 -> n2 = {n2}"
        elif n1 == 0:
            m1, n1 = x, 1
            action = f"n1 == 0 -> m1 = {x}, n1 = 1"
        elif n2 == 0:
            m2, n2 = x, 1
            action = f"n2 == 0 -> m2 = {x}, n2 = 1"
        else:
            n1, n2 = n1 - 1, n2 - 1
            action = f"khác cả hai -> gạch: n1 = {n1}, n2 = {n2}"
        print(f"  i={i}, x={x}: {action}; (m1,n1)=({m1},{n1}), (m2,n2)=({m2},{n2})")
    c1 = sum(1 for x in nums if x == m1)
    c2 = sum(1 for x in nums if x == m2)
    print(f"  xác minh: {m1} xuất hiện {c1} lần, {m2} xuất hiện {c2} lần (ngưỡng > {len(nums) // 3})")
    result = [m for m, c in ((m1, c1), (m2, c2)) if c > len(nums) // 3]
    print("  => đáp án:", result)
    return result


assert sorted(trace_boyer_moore([3, 1, 1, 2, 1, 2, 3, 2])) == [1, 2]
print()
assert sorted(trace_boyer_moore([3, 2, 3])) == [3]

nums = [3, 1, 1, 2, 1, 2, 3, 2] | n // 3 = 2
  i=0, x=3: n1 == 0 -> m1 = 3, n1 = 1; (m1,n1)=(3,1), (m2,n2)=(1,0)
  i=1, x=1: trùng m2 -> n2 = 1; (m1,n1)=(3,1), (m2,n2)=(1,1)
  i=2, x=1: trùng m2 -> n2 = 2; (m1,n1)=(3,1), (m2,n2)=(1,2)
  i=3, x=2: khác cả hai -> gạch: n1 = 0, n2 = 1; (m1,n1)=(3,0), (m2,n2)=(1,1)
  i=4, x=1: trùng m2 -> n2 = 2; (m1,n1)=(3,0), (m2,n2)=(1,2)
  i=5, x=2: n1 == 0 -> m1 = 2, n1 = 1; (m1,n1)=(2,1), (m2,n2)=(1,2)
  i=6, x=3: khác cả hai -> gạch: n1 = 0, n2 = 1; (m1,n1)=(2,0), (m2,n2)=(1,1)
  i=7, x=2: trùng m1 -> n1 = 1; (m1,n1)=(2,1), (m2,n2)=(1,1)
  xác minh: 2 xuất hiện 3 lần, 1 xuất hiện 3 lần (ngưỡng > 2)
  => đáp án: [2, 1]

nums = [3, 2, 3] | n // 3 = 1
  i=0, x=3: n1 == 0 -> m1 = 3, n1 = 1; (m1,n1)=(3,1), (m2,n2)=(1,0)
  i=1, x=2: n2 == 0 -> m2 = 2, n2 = 1; (m1,n1)=(3,1), (m2,n2)=(2,1)
  i=2, x=3: trùng m1 -> n1 = 2; (m1,n1)=(3,2), (m2,n2)=(2,1)
  xác minh: 3 xuất hiện 2 lần, 2 xuất hiện 1 lần (ngưỡng > 1)
  => đáp án: [3]


Bảng tóm tắt (`nums = [3,1,1,2,1,2,3,2]`):

| `i` | `x` | nhánh | `(m1,n1)` | `(m2,n2)` |
|---|---|---|---|---|
| 0 | 3 | `n1 == 0` | `(3,1)` | `(1,0)` |
| 1 | 1 | `x == m2` (giá trị khởi đầu `1`) | `(3,1)` | `(1,1)` |
| 2 | 1 | `x == m2` | `(3,1)` | `(1,2)` |
| 3 | 2 | khác cả hai: gạch | `(3,0)` | `(1,1)` |
| 4 | 1 | `x == m2` | `(3,0)` | `(1,2)` |
| 5 | 2 | `n1 == 0`: thay `m1` | `(2,1)` | `(1,2)` |
| 6 | 3 | khác cả hai: gạch | `(2,0)` | `(1,1)` |
| 7 | 2 | `x == m1` | `(2,1)` | `(1,1)` |

Hai ứng viên cuối cùng là `2` và `1`. Xác minh: `1` xuất hiện 3 lần, `2` xuất hiện 3 lần, đều `> 8 // 3 = 2`. Đáp án `[1, 2]`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 5 (Boyer–Moore, `O(1)` bộ nhớ, đáp ứng follow up). Đúng class `Solution`, đúng method `majorityElement(self, nums: List[int]) -> List[int]`.

In [16]:
from typing import List


class Solution:
    def majorityElement(self, nums: List[int]) -> List[int]:
        n1 = n2 = 0
        m1, m2 = 0, 1                       # hai ứng viên khác nhau (bộ đếm 0 nên chưa có tác dụng)
        for x in nums:
            if x == m1:
                n1 += 1
            elif x == m2:
                n2 += 1
            elif n1 == 0:
                m1, n1 = x, 1
            elif n2 == 0:
                m2, n2 = x, 1
            else:
                n1, n2 = n1 - 1, n2 - 1     # gạch một bản của ba giá trị khác nhau
        limit = len(nums) // 3
        return [m for m in (m1, m2) if nums.count(m) > limit]     # xác minh lượt hai


for nums, expected in examples:
    assert sorted(Solution().majorityElement(nums)) == sorted(expected)
print("PASS ví dụ đề cho class Solution")

PASS ví dụ đề cho class Solution


## 10. Vì sao thuật toán đúng

- **Tối đa 2 đáp án:** ba giá trị đều xuất hiện `> n/3` lần cho tổng `> n` phần tử, vô lý.
- **Bất biến "gạch ba giá trị khác nhau":** ở nhánh `else`, `x`, `m1`, `m2` đôi một khác nhau. Có thể xem như loại bỏ ba phần tử (một bản mỗi giá trị) ra khỏi mảng: phần mảng còn lại tiếp tục có cùng tập "đáp án" vì một giá trị `v` mất tối đa một bản trong ba phần tử bị loại; nếu `v` xuất hiện `> n/3` lần thì sau `t` lần loại (mỗi lần loại 3 phần tử) còn `> n/3 − t` bản trong `n − 3t` phần tử, tức vẫn `> (n − 3t)/3`.
- **Ứng viên sống sót:** vì mỗi lần gạch bỏ `3` phần tử khác nhau và `v` mất tối đa một bản, tối đa `⌊n/3⌋` lần gạch xảy ra; nếu `v` có `> n/3` bản thì còn ít nhất một bản không bị gạch, nên `v` là một trong hai ứng viên khi kết thúc lượt 1.
- **Lượt 2 bắt buộc:** hai ứng viên cuối chưa chắc thỏa `> n/3` (ví dụ `[1,2,3]` cho ứng viên nhưng đáp án rỗng). Đếm lại loại các ứng viên giả.
- **Thứ tự nhánh:** phải kiểm tra `x == m1` rồi `x == m2` **trước** khi xét `n1 == 0` hay `n2 == 0`. Nếu không, một giá trị đang là ứng viên thứ hai có thể bị nhận nhầm làm ứng viên thứ nhất khi `n1 == 0`, làm hai ứng viên bằng nhau và mất ứng viên thật.
- **Cách 4:** trên mảng đã sắp, `s[i] == s[i + n//3]` nghĩa là đoạn `[i, i + n//3]` toàn một giá trị, gồm `n//3 + 1` phần tử, đúng bằng ngưỡng `> n//3`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `d` là số giá trị khác nhau (`d ≤ n`).

- **Cách 1 (đếm bằng `count`):** Time `O(n·d)` (tệ nhất `O(n²)`), Space `O(d)`.
- **Cách 2 (`Counter`):** Time `O(n)` trung bình, Space `O(d)`.
- **Cách 3 (sắp xếp + đoạn):** Time `O(n log n)`, Space `O(n)`.
- **Cách 4 (`s[i] == s[i + n//3]`):** Time `O(n log n)`, Space `O(n)`.
- **Cách 5 (Boyer–Moore):** Time `O(n)`, Space `O(1)`.
- Trường hợp trung bình và xấu nhất của Cách 5 như nhau. `Counter` có thể về lý thuyết chậm khi băm va chạm nặng, nhưng với số nguyên thực tế là `O(n)`.

## 12. Bộ test

Ta gom các cách vào từ điển `impls`; `oracle` dựa vào `list.count` trên **mọi giá trị khác nhau** (Cách 1 viết lại, không dùng `Counter`, sắp xếp hay Boyer–Moore). So sánh **không phụ thuộc thứ tự**.

In [17]:
sol = Solution()
impls = {
    "bruteforce": majority_bruteforce,
    "counter": majority_counter,
    "sort_runs": majority_sort_runs,
    "sorted_window": majority_sorted_window,
    "boyer_moore": majority_boyer_moore,
    "submit": sol.majorityElement,
}


def oracle(nums):
    n = len(nums)
    return sorted(x for x in set(nums) if nums.count(x) > n // 3)


def run_all(nums):
    results = {}
    for name, fn in impls.items():
        original = list(nums)
        results[name] = sorted(fn(original))
        assert original == nums, (name, "đã sửa đầu vào")
    return results


for nums, expected in examples:
    assert oracle(nums) == sorted(expected)
    for name, got in run_all(nums).items():
        assert got == sorted(expected), (name, nums, got, expected)
print("PASS ví dụ của đề (mọi cách và oracle độc lập)")

PASS ví dụ của đề (mọi cách và oracle độc lập)


### 12.1. Các trường hợp biên

In [18]:
def check_all(nums, expected, label):
    assert oracle(nums) == sorted(expected), (label, "oracle", oracle(nums), expected)
    for name, got in run_all(nums).items():
        assert got == sorted(expected), (label, name, got, expected)
    print("PASS", label, "->", sorted(expected))


check_all([1], [1], "n = 1")
check_all([1, 2], [1, 2], "n = 2: ngưỡng 0, cả hai giá trị")
check_all([1, 1], [1], "n = 2, trùng")
check_all([1, 2, 3], [], "n = 3: ngưỡng 1, không giá trị nào xuất hiện hơn 1 lần")
check_all([1, 1, 2], [1], "n = 3: chỉ 1")
check_all([1, 2, 3, 4], [], "mọi giá trị khác nhau, n = 4 (ngưỡng 1)")
check_all([1, 1, 2, 2], [1, 2], "n = 4: hai giá trị xuất hiện 2 lần > 1")
check_all([2, 2, 1, 1, 1, 3, 4], [1], "n = 7, ngưỡng 2: chỉ 1 (3 lần); 2 có đúng 2 lần, không > 2")
check_all([-10**9, 10**9, -10**9, 10**9], [-10**9, 10**9], "giá trị biên +-10^9")
check_all([0, 0, 0, 1, 1, 1, 2], [0, 1], "có số 0 và hai giá trị hơn ngưỡng")
check_all([3, 1, 1, 2, 1, 2, 3, 2], [1, 2], "ví dụ trace")
check_all([5] * 50000, [5], "n = 5*10^4, toàn một giá trị")

PASS n = 1 -> [1]
PASS n = 2: ngưỡng 0, cả hai giá trị -> [1, 2]
PASS n = 2, trùng -> [1]
PASS n = 3: ngưỡng 1, không giá trị nào xuất hiện hơn 1 lần -> []
PASS n = 3: chỉ 1 -> [1]
PASS mọi giá trị khác nhau, n = 4 (ngưỡng 1) -> []
PASS n = 4: hai giá trị xuất hiện 2 lần > 1 -> [1, 2]
PASS n = 7, ngưỡng 2: chỉ 1 (3 lần); 2 có đúng 2 lần, không > 2 -> [1]
PASS giá trị biên +-10^9 -> [-1000000000, 1000000000]
PASS có số 0 và hai giá trị hơn ngưỡng -> [0, 1]
PASS ví dụ trace -> [1, 2]
PASS n = 5*10^4, toàn một giá trị -> [5]


### 12.2. Tính chất và phản ví dụ

In [19]:
random.seed(229)

# Tính chất 1: đáp án có tối đa 2 phần tử
for _ in range(500):
    n = random.randint(1, 40)
    nums = [random.randint(-3, 3) for _ in range(n)]
    assert len(sol.majorityElement(nums)) <= 2
print("PASS tính chất: tối đa 2 đáp án")

# Tính chất 2: thứ tự nums không ảnh hưởng, thêm hằng số vào mọi phần tử chỉ dịch đáp án
for _ in range(300):
    n = random.randint(1, 30)
    nums = [random.randint(-3, 3) for _ in range(n)]
    shuffled = nums[:]
    random.shuffle(shuffled)
    a = sorted(sol.majorityElement(nums))
    assert a == sorted(sol.majorityElement(shuffled))
    assert sorted(x + 100 for x in a) == sorted(sol.majorityElement([x + 100 for x in nums]))
print("PASS tính chất: hoán vị và dịch giá trị")

# Phản ví dụ 1: Boyer-Moore KHÔNG xác minh trả ứng viên giả
def sai_khong_xac_minh(nums):
    n1 = n2 = 0
    m1, m2 = 0, 1
    for x in nums:
        if x == m1:
            n1 += 1
        elif x == m2:
            n2 += 1
        elif n1 == 0:
            m1, n1 = x, 1
        elif n2 == 0:
            m2, n2 = x, 1
        else:
            n1, n2 = n1 - 1, n2 - 1
    return sorted({m1, m2})

print("[1,2,3] -> sai:", sai_khong_xac_minh([1, 2, 3]), "| đúng:", sol.majorityElement([1, 2, 3]))
assert sai_khong_xac_minh([1, 2, 3]) != []

PASS tính chất: tối đa 2 đáp án
PASS tính chất: hoán vị và dịch giá trị
[1,2,3] -> sai: [1, 2] | đúng: []


### 12.3. Đối chiếu ngẫu nhiên (có seed) với oracle độc lập

In [20]:
random.seed(2026)
for trial in range(3000):
    n = random.randint(1, 25)
    hi = random.choice([1, 2, 3, 5, 10])
    nums = [random.randint(-hi, hi) for _ in range(n)]
    expected = oracle(nums)
    for name, got in run_all(nums).items():
        assert got == expected, (name, nums, got, expected)

# mảng dài hơn, chỉ đối chiếu các cách nhanh (bruteforce O(n*d) vẫn dùng với ít giá trị khác nhau)
for trial in range(100):
    n = random.randint(500, 3000)
    hi = random.choice([2, 3, 5, 8])
    nums = [random.randint(-hi, hi) for _ in range(n)]
    expected = oracle(nums)
    for name, got in run_all(nums).items():
        assert got == expected, (name, n)
print("PASS 3100 input ngẫu nhiên: mọi cách khớp oracle độc lập")

PASS 3100 input ngẫu nhiên: mọi cách khớp oracle độc lập


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Nhầm với bài `> n/2` (chỉ giữ một ứng viên)

In [21]:
def sai_mot_ung_vien(nums):
    cand, cnt = None, 0
    for x in nums:
        if cnt == 0:
            cand, cnt = x, 1
        elif x == cand:
            cnt += 1
        else:
            cnt -= 1
    return [cand] if nums.count(cand) > len(nums) // 3 else []


print("[1,1,2,2] -> sai:", sai_mot_ung_vien([1, 1, 2, 2]), "| đúng:", sorted(Solution().majorityElement([1, 1, 2, 2])))

[1,1,2,2] -> sai: [1] | đúng: [1, 2]


**Vì sao sai:** với ngưỡng `n/3` có thể có **hai** đáp án. Một ứng viên không đủ: `[1,1,2,2]` có cả `1` và `2` (mỗi giá trị 2 lần `> 4 // 3 = 1`). Cần hai ứng viên và hai bộ đếm.

### Lỗi 2: Quên lượt xác minh

In [22]:
print("[1,2,3] -> sai:", sai_khong_xac_minh([1, 2, 3]), "| đúng:", Solution().majorityElement([1, 2, 3]))
print("[2,2,1,1,1,3,4] -> sai:", sai_khong_xac_minh([2, 2, 1, 1, 1, 3, 4]), "| đúng:", Solution().majorityElement([2, 2, 1, 1, 1, 3, 4]))

[1,2,3] -> sai: [1, 2] | đúng: []
[2,2,1,1,1,3,4] -> sai: [1, 2] | đúng: [1]


**Vì sao sai:** thuật toán chỉ cho **ứng viên**, không đảm bảo ứng viên thật sự xuất hiện `> n/3` lần. `[1,2,3]`: hai ứng viên cuối là `2,3` hoặc tương tự nhưng không giá trị nào xuất hiện hơn `1` lần. Lượt hai `count` loại các ứng viên giả.

### Lỗi 3: Dùng `>=` thay vì `>`

In [23]:
def sai_lon_hon_hoac_bang(nums):
    n = len(nums)
    return [x for x, c in Counter(nums).items() if c >= n // 3]


print("[1,2,3] -> sai:", sai_lon_hon_hoac_bang([1, 2, 3]), "| đúng:", Solution().majorityElement([1, 2, 3]))

[1,2,3] -> sai: [1, 2, 3] | đúng: []


**Vì sao sai:** đề yêu cầu "**nhiều hơn** `⌊n/3⌋` lần". `[1,2,3]`: `n // 3 = 1`, mỗi giá trị đúng 1 lần, không `> 1`; đáp án phải rỗng. Với `>=` ta trả cả ba.

### Lỗi 4: Sai thứ tự nhánh khi cập nhật ứng viên

In [24]:
def sai_thu_tu_nhanh(nums):
    n1 = n2 = 0
    m1, m2 = 0, 1
    for x in nums:
        if x == m1:
            n1 += 1
        elif n1 == 0:                       # sai: xét n1 == 0 trước khi kiểm tra x == m2
            m1, n1 = x, 1
        elif x == m2:
            n2 += 1
        elif n2 == 0:
            m2, n2 = x, 1
        else:
            n1, n2 = n1 - 1, n2 - 1
    limit = len(nums) // 3
    return sorted(m for m in {m1, m2} if nums.count(m) > limit)


bad = [3, 1, 1, 2, 1, 2, 3, 2]
print(bad, "-> sai:", sai_thu_tu_nhanh(bad), "| đúng:", sorted(Solution().majorityElement(bad)))

[3, 1, 1, 2, 1, 2, 3, 2] -> sai: [2] | đúng: [1, 2]


**Vì sao sai:** khi `x` đúng là ứng viên thứ hai (`x == m2`) nhưng `n1 == 0`, code sai đưa `x` vào vị trí `m1`, làm `m1` và `m2` trùng nhau (cùng giá trị) và cộng dồn phiếu sai. Phải kiểm tra **cả hai** điều kiện khớp (`x == m1`, `x == m2`) trước, rồi mới xét ô trống.

## 14. Mẹo làm nhanh & nhận dạng pattern

- "Phần tử xuất hiện hơn `n/k` lần" → có **tối đa `k − 1`** phần tử; Boyer–Moore mở rộng với `k − 1` ứng viên.
- Nếu không yêu cầu `O(1)` bộ nhớ thì `Counter` là cách ngắn, rõ nhất.
- **Luôn xác minh** ứng viên bằng lượt đếm thứ hai (trừ khi đề đảm bảo đáp án tồn tại như bài `> n/2` gốc).
- Trong mảng đã sắp, "xuất hiện ít nhất `m` lần" tương đương tồn tại `i` với `s[i] == s[i + m − 1]`.
- Sắp xếp, `groupby` hoặc `Counter` đều tìm được đoạn bằng nhau; chọn theo yêu cầu bộ nhớ.
- Khi test thuật toán "tìm tập đáp án" hãy **so sánh tập đã sắp** vì thứ tự không quan trọng.

## 15. Biến thể của bài

1. **`> n/2` (LeetCode 169):** một ứng viên Boyer–Moore.
2. **`> n/k` tổng quát:** `k − 1` ứng viên (dùng từ điển cỡ `k − 1`).
3. **Đếm có bao nhiêu giá trị** vượt ngưỡng (không cần liệt kê).
4. **`>= n/3`** hoặc ngưỡng khác (đổi điều kiện).
5. **Tìm phần tử xuất hiện nhiều nhất** (mode) — `Counter.most_common(1)`.

In [25]:
def majority_k(nums: list[int], k: int) -> list[int]:
    """Biến thể 2: mọi giá trị xuất hiện hơn n // k lần, Boyer-Moore tổng quát với k - 1 ứng viên."""
    cands = {}
    for x in nums:
        if x in cands:
            cands[x] += 1
        elif len(cands) < k - 1:
            cands[x] = 1
        else:                                       # gạch một bản của k giá trị khác nhau
            for key in list(cands):
                cands[key] -= 1
                if cands[key] == 0:
                    del cands[key]
    limit = len(nums) // k
    return [m for m in cands if nums.count(m) > limit]


def majority_half_element(nums):
    """Biến thể 1."""
    return majority_k(nums, 2)


def mode_value(nums):
    """Biến thể 5."""
    return Counter(nums).most_common(1)[0][0]


random.seed(15)
for _ in range(500):
    n = random.randint(1, 30)
    nums = [random.randint(-3, 3) for _ in range(n)]
    assert sorted(majority_k(nums, 3)) == oracle(nums)
    for k in (2, 4, 5):
        expected = sorted(x for x in set(nums) if nums.count(x) > n // k)
        assert sorted(majority_k(nums, k)) == expected, (nums, k)
assert majority_half_element([2, 2, 1, 1, 1, 2, 2]) == [2]
assert mode_value([1, 2, 2, 3]) == 2
print("PASS biến thể: n/k tổng quát, n/2, mode")

PASS biến thể: n/k tổng quát, n/2, mode


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `count` từng giá trị | mỗi giá trị quét lại mảng | `O(n·d)` | `O(d)` | hiểu đề; oracle (mảng nhỏ) |
| 2. `Counter` | đếm một lượt | `O(n)` | `O(d)` | **cách ngắn nhất**, nộp bài thường dùng |
| 3. Sắp xếp + đoạn | độ dài đoạn bằng nhau | `O(n log n)` | `O(n)` | không muốn dùng từ điển |
| 4. `s[i] == s[i + n//3]` | cửa sổ trên mảng đã sắp | `O(n log n)` | `O(n)` | một dòng ý tưởng gọn |
| 5. Boyer–Moore hai ứng viên | gạch ba giá trị khác nhau + xác minh | `O(n)` | `O(1)` | **follow up**, bộ nhớ hằng số |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `7 // 3`, `Counter([1, 1, 2])[1]`, `sorted({3, 1, 3})`, `Solution().majorityElement([2, 2, 2])`, `Solution().majorityElement([1, 2, 3])`.

**Bài 2 (tính tay).** Với `nums = [4, 4, 5, 5, 5, 6, 7, 4, 8]` (`n = 9`): tính ngưỡng `n // 3`, số lần mỗi giá trị và đáp án. Làm lại với `nums = [1, 2, 3, 1, 2, 3]`.

**Bài 3 (điền TODO).** Hoàn thành `majority_todo(nums)` theo Cách 2 (`Counter`).

In [26]:
def majority_todo(nums):
    # TODO 1: n = len(nums)
    # TODO 2: cnt = Counter(nums)
    # TODO 3: trả về list các giá trị x có cnt[x] > n // 3
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `majority_bm2(nums)` theo Cách 5 (Boyer–Moore hai ứng viên và lượt xác minh) mà không nhìn code mẫu. Kiểm tra bằng ba ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_majority(nums)` trả về **số lượng** giá trị xuất hiện hơn `n // 3` lần. (b) Viết `majority_quarter(nums)` trả về mọi giá trị xuất hiện hơn `n // 4` lần (tối đa 3 giá trị).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `//` làm tròn xuống; `Counter` đếm số lần; ngưỡng là `n // 3`. Bài 2: đếm số lần từng giá trị rồi so với ngưỡng.
- **Mức 2 (vừa):** Bài 1: `2`, `2`, `[1, 3]`, `[2]`, `[]`. Bài 2: `[4,4,5,5,5,6,7,4,8]`: ngưỡng `3`, số lần `4`: 3, `5`: 3, còn lại 1; không giá trị nào `> 3` nên đáp án `[]`. `[1,2,3,1,2,3]`: ngưỡng `2`, mỗi giá trị 2 lần (không `> 2`), đáp án `[]`. Bài 3: ba TODO trong comment.
- **Mức 3 (sâu):** Bài 4: tám dòng nhánh `if/elif` như mục 6 rồi `nums.count`. Bài 5(a): `len(Solution().majorityElement(nums))`. Bài 5(b): dùng `majority_k(nums, 4)` hoặc `Counter` với ngưỡng `n // 4`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1
assert 7 // 3 == 2
assert Counter([1, 1, 2])[1] == 2
assert sorted({3, 1, 3}) == [1, 3]
assert Solution().majorityElement([2, 2, 2]) == [2]
assert Solution().majorityElement([1, 2, 3]) == []

# Bài 2
assert Solution().majorityElement([4, 4, 5, 5, 5, 6, 7, 4, 8]) == []
assert 9 // 3 == 3
assert Solution().majorityElement([1, 2, 3, 1, 2, 3]) == []

# Bài 3
def majority_todo_ans(nums):
    n = len(nums)
    cnt = Counter(nums)
    return [x for x in cnt if cnt[x] > n // 3]

for nums, expected in examples:
    assert sorted(majority_todo_ans(nums)) == sorted(expected)

# Bài 4
def majority_bm2(nums):
    n1 = n2 = 0
    m1, m2 = 0, 1
    for x in nums:
        if x == m1:
            n1 += 1
        elif x == m2:
            n2 += 1
        elif n1 == 0:
            m1, n1 = x, 1
        elif n2 == 0:
            m2, n2 = x, 1
        else:
            n1, n2 = n1 - 1, n2 - 1
    return [m for m in (m1, m2) if nums.count(m) > len(nums) // 3]

for nums, expected in examples:
    assert sorted(majority_bm2(nums)) == sorted(expected)

# Bài 5(a), 5(b)
def count_majority(nums):
    return len(Solution().majorityElement(nums))

def majority_quarter(nums):
    n = len(nums)
    return [x for x, c in Counter(nums).items() if c > n // 4]

assert count_majority([1, 1, 1, 3, 3, 2, 2, 2]) == 2
assert sorted(majority_quarter([1, 1, 1, 2, 2, 2, 3, 3, 3, 4])) == [1, 2, 3]
random.seed(19)
for _ in range(200):
    nums = [random.randint(1, 4) for _ in range(random.randint(1, 20))]
    assert sorted(majority_quarter(nums)) == sorted(majority_k(nums, 4))
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (Boyer–Moore hai ứng viên + xác minh)

```python
class Solution:
    def majorityElement(self, nums: List[int]) -> List[int]:
        n1 = n2 = 0
        m1, m2 = 0, 1
        for x in nums:
            if x == m1:
                n1 += 1
            elif x == m2:
                n2 += 1
            elif n1 == 0:
                m1, n1 = x, 1
            elif n2 == 0:
                m2, n2 = x, 1
            else:
                n1, n2 = n1 - 1, n2 - 1
        return [m for m in (m1, m2) if nums.count(m) > len(nums) // 3]
```

**Checklist**

- [ ] Ngưỡng là `n // 3` và điều kiện là **`>`** (nhiều hơn), không phải `>=`?
- [ ] Có **tối đa 2** đáp án; dùng **hai** ứng viên/bộ đếm?
- [ ] Kiểm `x == m1` và `x == m2` **trước** khi xét `n1 == 0` / `n2 == 0`?
- [ ] Có **lượt xác minh** (đếm lại) các ứng viên, và khử trùng khi `m1 == m2`?
- [ ] Đã thử: `n = 1`, `n = 2`, `[1,2,3]` (rỗng), toàn một giá trị, giá trị âm/biên `10⁹`?
- [ ] Đã đối chiếu với `Counter` hoặc `count` trên input ngẫu nhiên (so sánh tập đã sắp)?
- [ ] Skeleton đúng `class Solution` / `def majorityElement(self, nums: List[int]) -> List[int]`?